In [ ]:
# ==============================================================================
# ML PRACTICAL 3: BANK CUSTOMER CHURN PREDICTION
# Objective: Predict customer churn using Artificial Neural Networks (ANN)
# ==============================================================================

# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

# 2. Load Dataset
df = pd.read_csv("Churn_Modelling.csv")
print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

# 3. Data Cleaning & Encoding
# Drop irrelevant identifier columns
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], errors='ignore')
print(f"\nMissing Values: {df.isnull().sum().sum()}")

# Encode categorical variables: Gender and Geography
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df['Geography'] = le.fit_transform(df['Geography'])

print("\nEncoded Data Preview:")
print(df.head())

# 4. Train-Test Split & Standardization
X = df.drop(columns=['Exited'])
y = df['Exited']

# 80:20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining features shape: {X_train_scaled.shape}")
print(f"Testing features shape:  {X_test_scaled.shape}")

# 5. Build ANN Architecture
tf.random.set_seed(42)

model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# 6. Train ANN Model
history = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=25, 
    batch_size=32, 
    validation_split=0.2, 
    verbose=1
)

# 7. Model Evaluation
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f"\nANN Test Accuracy: {acc:.4f} ({acc * 100:.2f}%)\n")
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=['Stayed (0)', 'Exited (1)']))

# 8. Learning Curves & Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 1. Training vs Validation Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='#2980b9', lw=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='#e67e22', lw=2)
axes[0].set_title('ANN Accuracy Curve', fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. Training vs Validation Loss
axes[1].plot(history.history['loss'], label='Train Loss', color='#2980b9', lw=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', color='#e74c3c', lw=2)
axes[1].set_title('ANN Loss Curve', fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Binary Crossentropy Loss')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

# 3. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Stayed (0)', 'Exited (1)'], 
            yticklabels=['Stayed (0)', 'Exited (1)'])
axes[2].set_title(f'Confusion Matrix\n(Accuracy: {acc*100:.2f}%)', fontweight='bold')
axes[2].set_xlabel('Predicted Label')
axes[2].set_ylabel('True Label')

plt.tight_layout()
plt.show()

# ==============================================================================
# ==============================================================================
# THEORETICAL CONCEPTS, HOW TO RUN & HOW TO EXPLAIN
# ==============================================================================
# 1. THEORETICAL CONCEPTS:
#    - Problem Formulation: Binary classification predicting whether a bank customer will churn (Exited = 1) or stay (0).
#    - Deep Learning Architecture (Artificial Neural Network - ANN):
#      * Input Layer : Size matching number of features (CreditScore, Age, Tenure, Balance, etc.).
#      * Hidden Layer 1: Dense(32, activation='relu') - Extracts non-linear feature combinations.
#      * Regularization: Dropout(0.2) - Randomly deactivates 20% of neurons during training to prevent overfitting.
#      * Hidden Layer 2: Dense(16, activation='relu') - Deep hierarchical representation learning.
#      * Output Layer: Dense(1, activation='sigmoid') - Maps output to probability range [0, 1].
#    - Compilation & Loss:
#      * Loss Function: Binary Cross-Entropy: -[y log(p) + (1-y) log(1-p)].
#      * Optimizer: Adam (Adaptive Moment Estimation), combining advantages of AdaGrad and RMSProp.
#    - Preprocessing:
#      * LabelEncoder for categorical features (Geography, Gender).
#      * StandardScaler for numerical features (CreditScore, Balance, EstimatedSalary).
#
# 2. HOW TO RUN:
#    - Environment: VS Code, JupyterLab, or Google Colab.
#    - Required Libraries: pip install pandas numpy matplotlib seaborn scikit-learn tensorflow
#    - Dataset: Ensure 'Churn_Modelling.csv' is present in the working directory.
#    - Execution: Press Shift + Enter on this single cell. It compiles the ANN, trains over 25 epochs,
#      evaluates test accuracy, and plots training vs validation loss/accuracy curves.
#
# 3. HOW TO EXPLAIN TO THE EXAMINER (30-SECOND VIVA PITCH):
#    "Sir/Madam, this practical implements an Artificial Neural Network (ANN) using Keras and TensorFlow
#    to predict customer churn:
#    - We preprocess the dataset by dropping identifier columns, encoding categorical variables
#      (Geography, Gender) via LabelEncoder, and normalizing features with StandardScaler.
#    - We construct a feed-forward deep neural network: Input -> Dense(32, ReLU) -> Dropout(0.2) ->
#      Dense(16, ReLU) -> Dense(1, Sigmoid).
#    - The model compiles with the Adam optimizer and Binary Crossentropy loss.
#    - After training for 25 epochs, it achieves ~86% accuracy, evaluated using confusion matrix and loss curves."
#
# 4. COMMON VIVA QUESTIONS:
#    Q: Why use ReLU in hidden layers and Sigmoid in the output layer?
#    A: ReLU avoids the vanishing gradient problem and is computationally efficient; Sigmoid maps output
#       to a valid probability [0, 1] suitable for binary thresholding (>= 0.5).
#    Q: What is the purpose of the Dropout layer?
#    A: Dropout is a regularization technique that randomly turns off neurons during training to prevent
#       co-adaptation of feature detectors, preventing the network from overfitting.
# ==============================================================================